<a href="https://colab.research.google.com/github/eleoncmd/Server/blob/master/recsys_init_subsys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# Чтение данных
ratings = pd.read_csv('/content/Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
books = pd.read_csv('/content/Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')

# Предварительный просмотр данных
print("Ratings dataset:")
print(ratings.head())
print("\nBooks dataset:")
print(books.head())

# Очистка и предобработка данных
# Удаляем ненужные столбцы
books_clean = books[['ISBN', 'Title']].copy()



Ratings dataset:
   User-ID        ISBN  Rating
0   276725  034545104X       0
1   276726  0155061224       5
2   276727  0446520802       0
3   276729  052165615X       3
4   276729  0521795028       6

Books dataset:
         ISBN                                              Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

                 Author  Year                Publisher  
0    Mark P. O. Morford  2002  Oxford University Press  
1  Richard Bruce Wright  2001    HarperFlamingo Canada  
2          Carlo D'Este  1991          HarperPerennial  
3      Gina Bari Kolata  1999     Farrar Straus Giroux  
4       E. J. W. Barber  1999   W. W. Norton & Company  


In [ ]:
# Фильтрация пользователей и книг с малым количеством оценок
users_votes = ratings.groupby('User-ID')['Rating'].agg('count')
books_votes = ratings.groupby('ISBN')['Rating'].agg('count')

In [ ]:
# Создаем фильтры
user_mask = users_votes[users_votes > 50].index
book_mask = books_votes[books_votes > 10].index

In [ ]:
# Применяем фильтры
ratings_filtered = ratings[
    (ratings['User-ID'].isin(user_mask)) &
    (ratings['ISBN'].isin(book_mask))
]

In [ ]:
# Создание матрицы предпочтений
user_item_matrix = ratings_filtered.pivot_table(
    index='ISBN',
    columns='User-ID',
    values='Rating'
)

In [ ]:
# Заполняем пропущенные значения не нулями, а средними оценками
# Сначала средняя оценка по книге
book_means = user_item_matrix.mean(axis=1)
# Затем средняя оценка пользователя
user_means = user_item_matrix.mean(axis=0)

In [ ]:
# Заполняем пропуски: сначала средним по книге, затем средним по пользователю, затем общим средним
user_item_matrix_filled = user_item_matrix.apply(lambda row: row.fillna(book_means[row.name]), axis=1)
user_item_matrix_filled = user_item_matrix_filled.apply(lambda col: col.fillna(user_means[col.name]), axis=0)
user_item_matrix_filled = user_item_matrix_filled.fillna(user_item_matrix_filled.mean().mean())


In [ ]:
# Округляем значения до одного знака после запятой
user_item_matrix_filled = user_item_matrix_filled.round(1)

In [ ]:
print(f"\nUser-item matrix shape: {user_item_matrix_filled.shape}")
print("\nПервые 5 строк матрицы (первые 5 столбцов):")
print(user_item_matrix_filled.iloc[:5, :5])



User-item matrix shape: (16154, 3354)

Первые 5 строк матрицы (первые 5 столбцов):
User-ID     243  254  507  626  638
ISBN                               
000000000   1.0  1.0  1.0  1.0  1.0
0002005018  3.8  3.8  3.8  3.8  3.8
0002251760  3.9  3.9  3.9  3.9  3.9
0002259001  4.3  4.3  4.3  4.3  4.3
0002259834  5.5  5.5  5.5  5.5  5.5


In [ ]:
# Преобразуем в разреженный формат для эффективности
csr_data = csr_matrix(user_item_matrix_filled.values)
print(f"\nCSR matrix shape: {csr_data.shape}")
print("Первые 2 строки CSR матрицы (первые 5 столбцов):")
print(csr_data[:2, :5].toarray())



CSR matrix shape: (16154, 3354)
Первые 2 строки CSR матрицы (первые 5 столбцов):
[[1.  1.  1.  1.  1. ]
 [3.8 3.8 3.8 3.8 3.8]]


In [ ]:
# Сбрасываем индекс для удобства
user_item_matrix_filled = user_item_matrix_filled.rename_axis(None, axis=1).reset_index()
print("\nМатрица после сброса индекса (первые 3 строки):")
print(user_item_matrix_filled.head(3))


Матрица после сброса индекса (первые 3 строки):
         ISBN  243  254  507  626  638  643  741  882  929  ...  277928  \
0   000000000  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...     1.0   
1  0002005018  3.8  3.8  3.8  3.8  3.8  3.8  3.8  3.8  3.8  ...     3.8   
2  0002251760  3.9  3.9  3.9  3.9  3.9  3.9  3.9  3.9  3.9  ...     3.9   

   277965  278026  278137  278144  278188  278418  278582  278633  278843  
0     1.0     1.0     1.0     1.0     1.0     1.0     1.0     1.0     1.0  
1     3.8     3.8     3.8     3.8     3.8     3.8     3.8     3.8     3.8  
2     3.9     3.9     3.9     3.9     3.9     3.9     3.9     3.9     3.9  

[3 rows x 3355 columns]


In [ ]:
# Проверка на наличие NaN
print(f"\nКоличество NaN в матрице: {user_item_matrix_filled.isna().sum().sum()}")


Количество NaN в матрице: 0


In [1]:
# Профиль пользователя, товара. Модели функционирования.
# Вычисление похожести, близости и соответствия.
print("Профиль пользователя, товара. Модели функционирования.")
print("Вычисление похожести, близости и соответствия.")

Профиль пользователя, товара. Модели функционирования.
Вычисление похожести, близости и соответствия.


In [ ]:
# Создаем модель k-ближайших соседей с использованием косинусного расстояния
print("\n1. Создание модели k-ближайших соседей (KNN):")

# Используем параметры:
# - metric='cosine': косинусное сходство
# - algorithm='brute': полный перебор (brute force search)
# - n_neighbors=20: количество соседей для обучения
# - n_jobs=-1: использовать все доступные ядра процессора
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)

print("Параметры модели KNN:")
print(f"  Метрика расстояния: {knn.metric}")
print(f"  Алгоритм поиска: {knn.algorithm}")
print(f"  Количество соседей: {knn.n_neighbors}")
print(f"  Количество используемых ядер: {knn.n_jobs}")


1. Создание модели k-ближайших соседей (KNN):
Параметры модели KNN:
  Метрика расстояния: cosine
  Алгоритм поиска: brute
  Количество соседей: 20
  Количество используемых ядер: -1


In [ ]:
# Обучаем модель на наших данных (в формате CSR)
print("\n2. Обучение модели на матрице предпочтений...")
knn.fit(csr_data)

print("Модель успешно обучена!")
print("\nОбученная модель KNN:")
print(knn)


2. Обучение модели на матрице предпочтений...
Модель успешно обучена!

Обученная модель KNN:
NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=20)


In [ ]:
# Проверяем работу модели на нескольких примерах
print("\n3. Проверка работы модели:")


3. Проверка работы модели:


In [ ]:
# Создаем функцию для безопасного получения названия книги
def get_book_title(isbn):
    """Безопасно получает название книги по ISBN"""
    try:
        book_info = books_clean[books_clean['ISBN'] == isbn]
        if not book_info.empty:
            title = book_info['Title'].values[0]
            return title[:80] + "..." if len(title) > 80 else title
    except Exception:
        pass
    return "Название не найдено"

In [ ]:
# Берем несколько случайных книг для тестирования, но проверяем что они есть в books_clean
available_books = user_item_matrix_filled['ISBN'].isin(books_clean['ISBN'])
valid_indices = user_item_matrix_filled[available_books].index.tolist()

if len(valid_indices) >= 3:
    test_indices = [valid_indices[0], valid_indices[min(10, len(valid_indices)-1)],
                   valid_indices[min(50, len(valid_indices)-1)]]
    print(f"Тестируем на книгах с индексами: {test_indices}")

    for idx in test_indices:
        # Получаем ISBN книги по индексу
        isbn = user_item_matrix_filled.iloc[idx]['ISBN']

        # Получаем название книги
        title = get_book_title(isbn)

        print(f"\n  Тестируем книгу #{idx}:")
        print(f"    ISBN: {isbn}")
        print(f"    Название: {title}")

        try:
            # Находим k ближайших соседей
            distances, indices = knn.kneighbors(csr_data[idx], n_neighbors=6)  # 5 соседей + сама книга

            print(f"    Найдено {len(indices[0])-1} ближайших соседей:")
            for i, (neighbor_idx, distance) in enumerate(zip(indices[0], distances[0])):
                # Пропускаем самого себя (расстояние 0)
                if i == 0:
                    continue
                if i > 5:  # Показываем только первые 5 соседей
                    break

                # Получаем информацию о соседней книге
                neighbor_isbn = user_item_matrix_filled.iloc[neighbor_idx]['ISBN']
                neighbor_title = get_book_title(neighbor_isbn)

                print(f"      {i}. Расстояние: {distance:.4f}")
                print(f"         ISBN: {neighbor_isbn}")
                print(f"         Название: {neighbor_title}")
        except Exception as e:
            print(f"    Ошибка при поиске соседей: {e}")
else:
    print("Недостаточно книг для тестирования!")


Тестируем на книгах с индексами: [1, 30, 77]

  Тестируем книгу #1:
    ISBN: 0002005018
    Название: Clara Callan
    Найдено 5 ближайших соседей:
      1. Расстояние: 0.0011
         ISBN: 2277241202
         Название: L' Alchimiste
      2. Расстояние: 0.0011
         ISBN: 0345450175
         Название: Balance of Power
      3. Расстояние: 0.0012
         ISBN: 0722532938
         Название: Название не найдено
      4. Расстояние: 0.0012
         ISBN: 9722319345
         Название: Название не найдено
      5. Расстояние: 0.0012
         ISBN: 8472230082
         Название: Relato de un nÃÂ¡ufrago

  Тестируем книгу #30:
    ISBN: 0006543545
    Название: The bookshop
    Найдено 5 ближайших соседей:
      1. Расстояние: 0.0022
         ISBN: 9770390107900
         Название: Название не найдено
      2. Расстояние: 0.0022
         ISBN: 9724113361
         Название: Название не найдено
      3. Расстояние: 0.0022
         ISBN: 8881124254
         Название: Название не найдено
   

In [ ]:
# Создаем функцию для получения рекомендаций (будет использоваться в следующей лабораторной)
print("\n4. Создание функции для получения рекомендаций:")

def get_book_recommendations(isbn, k=10):
    """
    Функция для получения рекомендаций книг на основе ISBN

    Аргументы:
    - isbn: ISBN книги, для которой ищем рекомендации
    - k: количество рекомендаций (по умолчанию 10)

    Возвращает:
    - DataFrame с рекомендациями (название книги и расстояние)
    """
    try:
        # Проверяем, есть ли книга в матрице
        book_in_matrix = user_item_matrix_filled[user_item_matrix_filled['ISBN'] == isbn]
        if book_in_matrix.empty:
            print(f"Книга с ISBN {isbn} не найдена в матрице предпочтений.")
            return pd.DataFrame()

        # Находим индекс книги в матрице
        book_idx = book_in_matrix.index[0]

        # Находим k+1 ближайших соседей (включая саму книгу)
        distances, indices = knn.kneighbors(csr_data[book_idx], n_neighbors=k+1)

        # Преобразуем результаты в список
        indices_list = indices.squeeze().tolist()
        distances_list = distances.squeeze().tolist()

        # Объединяем индексы и расстояния
        indices_distances = list(zip(indices_list, distances_list))

        # Сортируем по расстоянию (по возрастанию)
        indices_distances_sorted = sorted(indices_distances, key=lambda x: x[1])

        # Убираем саму книгу (первый элемент)
        indices_distances_sorted = indices_distances_sorted[1:]

        # Создаем список для рекомендаций
        recommendations = []

        # Собираем информацию о рекомендуемых книгах
        for idx, dist in indices_distances_sorted[:k]:
            neighbor_isbn = user_item_matrix_filled.iloc[idx]['ISBN']

            # Получаем название книги
            title = get_book_title(neighbor_isbn)
            recommendations.append({
                'ISBN': neighbor_isbn,
                'Title': title,
                'Distance': round(dist, 4)
            })

        # Преобразуем в DataFrame
        if recommendations:
            return pd.DataFrame(recommendations, index=range(1, len(recommendations) + 1))
        else:
            return pd.DataFrame()

    except Exception as e:
        print(f"Ошибка при получении рекомендаций: {e}")
        return pd.DataFrame()

print("Функция get_book_recommendations успешно создана!")
print("Аргументы функции: ISBN книги и количество рекомендаций (по умолчанию 10)")


4. Создание функции для получения рекомендаций:
Функция get_book_recommendations успешно создана!
Аргументы функции: ISBN книги и количество рекомендаций (по умолчанию 10)


In [ ]:
# Тестируем функцию на примере
print("\n5. Тестирование функции рекомендаций на примере:")

# Ищем первую книгу, которая есть и в матрице, и в books_clean
available_isbns = user_item_matrix_filled[available_books]['ISBN']
if not available_isbns.empty:
    test_isbn = available_isbns.iloc[0]
    test_book_title = get_book_title(test_isbn)

    print(f"Тестируем рекомендации для книги:")
    print(f"  ISBN: {test_isbn}")
    print(f"  Название: {test_book_title}")

    # Получаем рекомендации
    recommendations = get_book_recommendations(test_isbn, k=5)

    if not recommendations.empty:
        print(f"\nПолучено {len(recommendations)} рекомендаций:")
        print(recommendations[['Title', 'Distance']].head())
    else:
        print("Не удалось получить рекомендации")
else:
    print("Нет доступных книг для тестирования!")


5. Тестирование функции рекомендаций на примере:
Тестируем рекомендации для книги:
  ISBN: 0002005018
  Название: Clara Callan

Получено 5 рекомендаций:
                      Title  Distance
1             L' Alchimiste    0.0011
2          Balance of Power    0.0011
3       Название не найдено    0.0012
4       Название не найдено    0.0012
5  Relato de un nÃÂ¡ufrago    0.0012


In [ ]:
# Анализ данных о сходстве
print("\n6. Анализ данных о сходстве книг:")

# Вычисляем среднее расстояние между ближайшими соседями
print("Вычисляем среднее расстояние между ближайшими соседями...")

sample_size = min(100, len(user_item_matrix_filled))
sample_indices = np.random.choice(len(user_item_matrix_filled), sample_size, replace=False)

total_distances = 0
count = 0

for idx in sample_indices:
    try:
        distances, _ = knn.kneighbors(csr_data[idx], n_neighbors=2)  # 1 ближайший сосед + сама книга
        if len(distances[0]) > 1:
            total_distances += distances[0][1]  # Берем расстояние до ближайшего соседа
            count += 1
    except Exception:
        continue

if count > 0:
    avg_distance = total_distances / count
    print(f"Среднее косинусное расстояние до ближайшего соседа: {avg_distance:.4f}")
    print(f"Средняя похожесть: {(1 - avg_distance):.4f}")

    if avg_distance < 0.3:
        print("Вывод: Книги в датасете имеют высокую степень похожести")
    elif avg_distance < 0.6:
        print("Вывод: Книги в датасете имеют среднюю степень похожести")
    else:
        print("Вывод: Книги в датасете имеют низкую степень похожести")
else:
    print("Не удалось вычислить среднее расстояние")


6. Анализ данных о сходстве книг:
Вычисляем среднее расстояние между ближайшими соседями...
Среднее косинусное расстояние до ближайшего соседа: 0.0378
Средняя похожесть: 0.9622
Вывод: Книги в датасете имеют высокую степень похожести


In [ ]:
# Сохраняем модель для использования в следующих лабораторных работах
print("\n7. Сохранение результатов:")

# Сохраняем обученную модель и подготовленные данные
import pickle
import os


7. Сохранение результатов:


In [ ]:
# Создаем папку для сохранения моделей
os.makedirs('models', exist_ok=True)

In [ ]:
# Сохраняем модель KNN
with open('models/knn_model.pkl', 'wb') as f:
    pickle.dump(knn, f)

In [ ]:
# Сохраняем матрицу предпочтений
with open('models/user_item_matrix.pkl', 'wb') as f:
    pickle.dump(user_item_matrix_filled, f)

In [ ]:
# Сохраняем CSR матрицу
with open('models/csr_data.pkl', 'wb') as f:
    pickle.dump(csr_data, f)

In [ ]:
# Сохраняем информацию о книгах
with open('models/books_clean.pkl', 'wb') as f:
    pickle.dump(books_clean, f)

In [ ]:
print("Модель и данные успешно сохранены в папке 'models':")
print("  - knn_model.pkl: обученная модель KNN")
print("  - user_item_matrix.pkl: матрица предпочтений")
print("  - csr_data.pkl: данные в формате CSR")
print("  - books_clean.pkl: информация о книгах")

Модель и данные успешно сохранены в папке 'models':
  - knn_model.pkl: обученная модель KNN
  - user_item_matrix.pkl: матрица предпочтений
  - csr_data.pkl: данные в формате CSR
  - books_clean.pkl: информация о книгах


In [ ]:
# Вывод итоговой информации
print("\n" + "=" * 80)
print("ИТОГИ ЛАБОРАТОРНОЙ РАБОТЫ №3:")
print("=" * 80)
print("1. Модель K-ближайших соседей успешно обучена с параметрами:")
print(f"   - Метрика: косинусное расстояние")
print(f"   - Алгоритм: полный перебор")
print(f"   - Количество соседей: 20")
print(f"   - Размер обучающей выборки: {csr_data.shape[0]} книг, {csr_data.shape[1]} пользователей")



ИТОГИ ЛАБОРАТОРНОЙ РАБОТЫ №3:
1. Модель K-ближайших соседей успешно обучена с параметрами:
   - Метрика: косинусное расстояние
   - Алгоритм: полный перебор
   - Количество соседей: 20
   - Размер обучающей выборки: 16154 книг, 3354 пользователей


In [ ]:
print("\n2. Создана функция get_book_recommendations для получения рекомендаций")
print("   на основе ISBN книги.")


2. Создана функция get_book_recommendations для получения рекомендаций
   на основе ISBN книги.


In [ ]:
print("\n3. Проведен анализ сходства книг в датасете")


3. Проведен анализ сходства книг в датасете


In [ ]:
print("\n4. Все данные и модель сохранены для использования")
print("   в следующих лабораторных работах.")


4. Все данные и модель сохранены для использования
   в следующих лабораторных работах.


In [ ]:
# Анализ данных о сходстве
print("\n6. Анализ данных о сходстве книг:")

# Вычисляем среднее расстояние между ближайшими соседями
print("Вычисляем среднее расстояние между ближайшими соседями...")


6. Анализ данных о сходстве книг:
Вычисляем среднее расстояние между ближайшими соседями...


In [ ]:
# ============================================
# ЛАБОРАТОРНАЯ РАБОТА №4
# Разработка алгоритмов функционирования рекомендательной системы
# ============================================

print("=" * 80)
print("ЛАБОРАТОРНАЯ РАБОТА №4")
print("Разработка алгоритмов функционирования рекомендательной системы")
print("=" * 80)

ЛАБОРАТОРНАЯ РАБОТА №4
Разработка алгоритмов функционирования рекомендательной системы


In [ ]:
import pickle
import pandas as pd
import numpy as np

try:
    # Загружаем модель KNN
    with open('models/knn_model.pkl', 'rb') as f:
        knn = pickle.load(f)

    # Загружаем матрицу предпочтений
    with open('models/user_item_matrix.pkl', 'rb') as f:
        user_item_matrix_filled = pickle.load(f)

    # Загружаем данные о книгах
    with open('models/books_clean.pkl', 'rb') as f:
        books_clean = pickle.load(f)

    # Загружаем CSR матрицу
    with open('models/csr_data.pkl', 'rb') as f:
        csr_data = pickle.load(f)

    print("Данные успешно загружены!")
    print(f"Размер матрицы предпочтений: {user_item_matrix_filled.shape}")
    print(f"Количество книг в каталоге: {len(books_clean)}")

except Exception as e:
    print(f"Ошибка при загрузке данных: {e}")
    print("Пожалуйста, убедитесь, что вы выполнили лабораторную работу №3")
    exit()

Данные успешно загружены!
Размер матрицы предпочтений: (16154, 3355)
Количество книг в каталоге: 271379


In [ ]:
# Функция для безопасного получения названия книги по ISBN
def get_book_title(isbn):
    """Безопасно получает название книги по ISBN"""
    try:
        book_info = books_clean[books_clean['ISBN'] == isbn]
        if not book_info.empty:
            title = book_info['Title'].values[0]
            return title
    except Exception:
        pass
    return "Название не найдено"


In [ ]:
# Функция для поиска книги по названию (части названия)
def find_books_by_title(search_word, max_results=5):
    """
    Поиск книг по названию или его части

    Аргументы:
    - search_word: часть названия книги для поиска
    - max_results: максимальное количество результатов

    Возвращает:
    - DataFrame с найденными книгами
    """
    try:
        # Ищем книги, содержащие указанную строку в названии
        search_result = books_clean[books_clean['Title'].str.contains(
            search_word, case=False, na=False)]

        if not search_result.empty:
            print(f"Найдено {len(search_result)} книг, содержащих '{search_word}' в названии")
            return search_result.head(max_results)
        else:
            print(f"Книг, содержащих '{search_word}' в названии, не найдено")
            return pd.DataFrame()

    except Exception as e:
        print(f"Ошибка при поиске книг: {e}")
        return pd.DataFrame()

In [ ]:
# Улучшенная функция для получения рекомендаций
def get_enhanced_recommendations(isbn, k=10):
    """
    Улучшенная функция для получения рекомендаций книг на основе ISBN

    Аргументы:
    - isbn: ISBN книги, для которой ищем рекомендации
    - k: количество рекомендаций (по умолчанию 10)

    Возвращает:
    - DataFrame с рекомендациями (ISBN, название книги, расстояние)
    """
    try:
        # Проверяем, есть ли книга в матрице
        book_in_matrix = user_item_matrix_filled[user_item_matrix_filled['ISBN'] == isbn]
        if book_in_matrix.empty:
            print(f"Книга с ISBN {isbn} не найдена в матрице предпочтений.")
            return pd.DataFrame()

        # Находим индекс книги в матрице
        book_idx = book_in_matrix.index[0]

        # Находим k+1 ближайших соседей (включая саму книгу)
        distances, indices = knn.kneighbors(csr_data[book_idx], n_neighbors=k+1)

        # Преобразуем результаты в списки
        indices_list = indices.squeeze().tolist()
        distances_list = distances.squeeze().tolist()

        # Объединяем индексы и расстояния
        indices_distances = list(zip(indices_list, distances_list))

        # Сортируем по расстоянию (по возрастанию)
        indices_distances_sorted = sorted(indices_distances, key=lambda x: x[1])

        # Убираем саму книгу (первый элемент)
        indices_distances_sorted = indices_distances_sorted[1:]

        # Создаем список для рекомендаций
        recommendations = []

        # Собираем информацию о рекомендуемых книгах
        for idx, dist in indices_distances_sorted[:k]:
            neighbor_isbn = user_item_matrix_filled.iloc[idx]['ISBN']

            # Получаем название книги
            title = get_book_title(neighbor_isbn)

            # Определяем категорию рекомендации на основе расстояния
            if dist < 0.1:
                category = "Очень похожая"
            elif dist < 0.3:
                category = "Похожая"
            elif dist < 0.5:
                category = "Умеренно похожая"
            else:
                category = "Отдаленно похожая"

            recommendations.append({
                'ISBN': neighbor_isbn,
                'Title': title,
                'Distance': round(dist, 4),
                'Category': category
            })

        # Преобразуем в DataFrame
        if recommendations:
            df = pd.DataFrame(recommendations, index=range(1, len(recommendations) + 1))
            return df
        else:
            return pd.DataFrame()

    except Exception as e:
        print(f"Ошибка при получении рекомендаций: {e}")
        return pd.DataFrame()

In [ ]:
# Функция для полного цикла рекомендаций
def recommendation_pipeline(search_word, k=10):
    """
    Полный цикл рекомендаций: поиск книги -> получение рекомендаций

    Аргументы:
    - search_word: часть названия книги для поиска
    - k: количество рекомендаций

    Возвращает:
    - DataFrame с рекомендациями или пустой DataFrame
    """
    print(f"\n{'='*60}")
    print(f"ПОИСК РЕКОМЕНДАЦИЙ ДЛЯ: '{search_word}'")
    print('='*60)

    # Шаг 1: Поиск книги
    found_books = find_books_by_title(search_word)

    if found_books.empty:
        return pd.DataFrame()

    # Показываем найденные книги
    print(f"\nНайденные книги:")
    for i, (_, book) in enumerate(found_books.iterrows(), 1):
        print(f"{i}. ISBN: {book['ISBN']}")
        print(f"   Название: {book['Title']}")

    # Шаг 2: Выбор книги для рекомендаций (берем первую)
    selected_isbn = found_books.iloc[0]['ISBN']
    selected_title = found_books.iloc[0]['Title']

    print(f"\nВыбрана книга для рекомендаций:")
    print(f"  ISBN: {selected_isbn}")
    print(f"  Название: {selected_title}")

    # Шаг 3: Получение рекомендаций
    print(f"\nПолучение рекомендаций...")
    recommendations = get_enhanced_recommendations(selected_isbn, k)

    if recommendations.empty:
        print("Не удалось получить рекомендации.")
        return pd.DataFrame()

    # Шаг 4: Вывод результатов
    print(f"\nРЕКОМЕНДАЦИИ ДЛЯ КНИГИ:")
    print(f"'{selected_title}'")
    print('-'*60)

    # Группируем рекомендации по категориям
    categories = recommendations['Category'].unique()

    for category in categories:
        cat_books = recommendations[recommendations['Category'] == category]
        print(f"\n{category} ({len(cat_books)} книг):")

        for _, book in cat_books.iterrows():
            print(f"  • {book['Title']} (расстояние: {book['Distance']})")

    return recommendations

In [ ]:
# Основная часть работы
print("\n2. Тестирование алгоритма рекомендаций на различных примерах:")

# Пример 1: Поиск рекомендаций для книги по названию
print("\nПример 1: Тестирование полного цикла рекомендаций")
test_search_terms = ['Harry Potter', 'Lord of the Rings', 'Da Vinci', 'Pride and Prejudice']

for search_term in test_search_terms:
    try:
        recommendations = recommendation_pipeline(search_term, k=8)

        if not recommendations.empty:
            # Сохраняем результаты в файл
            filename = f"рекомендации_{search_term.replace(' ', '_')}.csv"
            recommendations.to_csv(filename, encoding='utf-8')
            print(f"\nРезультаты сохранены в файл: {filename}")
    except Exception as e:
        print(f"Ошибка при обработке '{search_term}': {e}")


2. Тестирование алгоритма рекомендаций на различных примерах:

Пример 1: Тестирование полного цикла рекомендаций

ПОИСК РЕКОМЕНДАЦИЙ ДЛЯ: 'Harry Potter'
Найдено 120 книг, содержащих 'Harry Potter' в названии

Найденные книги:
1. ISBN: 0767908473
   Название: The Sorcerer's Companion: A Guide to the Magical World of Harry Potter
2. ISBN: 059035342X
   Название: Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))
3. ISBN: 0590353403
   Название: Harry Potter and the Sorcerer's Stone (Book 1)
4. ISBN: 0439064872
   Название: Harry Potter and the Chamber of Secrets (Book 2)
5. ISBN: 0439136350
   Название: Harry Potter and the Prisoner of Azkaban (Book 3)

Выбрана книга для рекомендаций:
  ISBN: 0767908473
  Название: The Sorcerer's Companion: A Guide to the Magical World of Harry Potter

Получение рекомендаций...

РЕКОМЕНДАЦИИ ДЛЯ КНИГИ:
'The Sorcerer's Companion: A Guide to the Magical World of Harry Potter'
------------------------------------------------------------

Очен

In [ ]:
# Пример 2: Анализ качества рекомендаций
print("\n\n3. Анализ качества рекомендаций:")

def evaluate_recommendations_quality(isbn, recommendations_df):
    """
    Простой анализ качества рекомендаций

    Аргументы:
    - isbn: ISBN исходной книги
    - recommendations_df: DataFrame с рекомендациями

    Возвращает:
    - Словарь с метриками качества
    """
    if recommendations_df.empty:
        return {}

    # Получаем жанр исходной книги (если есть в данных)
    try:
        # В реальной системе здесь была бы проверка жанра
        # Для упрощения оцениваем только по расстоянию
        avg_distance = recommendations_df['Distance'].mean()
        min_distance = recommendations_df['Distance'].min()
        max_distance = recommendations_df['Distance'].max()

        metrics = {
            'Количество рекомендаций': len(recommendations_df),
            'Среднее расстояние': round(avg_distance, 4),
            'Минимальное расстояние': round(min_distance, 4),
            'Максимальное расстояние': round(max_distance, 4),
            'Диапазон расстояний': round(max_distance - min_distance, 4)
        }

        # Качественная оценка
        if avg_distance < 0.2:
            metrics['Качество'] = 'Отличное'
        elif avg_distance < 0.4:
            metrics['Качество'] = 'Хорошее'
        elif avg_distance < 0.6:
            metrics['Качество'] = 'Удовлетворительное'
        else:
            metrics['Качество'] = 'Низкое'

        return metrics

    except Exception as e:
        print(f"Ошибка при анализе качества: {e}")
        return {}



3. Анализ качества рекомендаций:


In [ ]:
# Тестируем на конкретном примере
print("\nОценка качества на примере книги:")
test_isbn = user_item_matrix_filled.iloc[1]['ISBN']
test_title = get_book_title(test_isbn)
print(f"Книга: {test_title}")


Оценка качества на примере книги:
Книга: Clara Callan


In [ ]:
test_recommendations = get_enhanced_recommendations(test_isbn, k=10)
if not test_recommendations.empty:
    quality_metrics = evaluate_recommendations_quality(test_isbn, test_recommendations)

    print("\nМетрики качества рекомендаций:")
    for metric, value in quality_metrics.items():
        print(f"  {metric}: {value}")


Метрики качества рекомендаций:
  Количество рекомендаций: 10
  Среднее расстояние: 0.0012
  Минимальное расстояние: 0.0011
  Максимальное расстояние: 0.0012
  Диапазон расстояний: 0.0001
  Качество: Отличное


In [ ]:
# Пример 3: Сравнение рекомендаций для разных книг
print("\n\n4. Сравнение рекомендаций для разных книг:")

def compare_recommendations(isbn_list, k=5):
    """
    Сравнивает рекомендации для нескольких книг

    Аргументы:
    - isbn_list: список ISBN книг для сравнения
    - k: количество рекомендаций для каждой книги
    """
    results = {}

    for isbn in isbn_list[:3]:  # Ограничиваемся тремя книгами
        title = get_book_title(isbn)
        print(f"\nКнига: {title}")

        recommendations = get_enhanced_recommendations(isbn, k)
        if not recommendations.empty:
            results[isbn] = {
                'title': title,
                'recommendations': recommendations,
                'avg_distance': recommendations['Distance'].mean()
            }

            print(f"  Среднее расстояние: {recommendations['Distance'].mean():.4f}")
            print(f"  Лучшая рекомендация: {recommendations.iloc[1]['Title']}")

    return results



4. Сравнение рекомендаций для разных книг:


In [ ]:
# Выбираем несколько книг для сравнения
available_isbns = user_item_matrix_filled['ISBN'].head(10).tolist()
comparison_results = compare_recommendations(available_isbns, k=3)

# Заключительная часть
print("\n" + "="*80)
print("5. Сохранение результатов и модели:")


Книга: Название не найдено
  Среднее расстояние: 0.0032
  Лучшая рекомендация: Название не найдено

Книга: Clara Callan
  Среднее расстояние: 0.0011
  Лучшая рекомендация: L' Alchimiste

Книга: The Forgetting Room: A Fiction (Byzantium Book)
  Среднее расстояние: 0.0018
  Лучшая рекомендация: Название не найдено

5. Сохранение результатов и модели:


In [ ]:
# Сохраняем улучшенную модель и функции
import os

# Создаем папку для сохранения результатов
os.makedirs('lab4_results', exist_ok=True)

In [ ]:
# Сохраняем примеры рекомендаций
if 'comparison_results' in locals() and comparison_results:
    with open('lab4_results/comparison_results.pkl', 'wb') as f:
        pickle.dump(comparison_results, f)

# Сохраняем тестовые рекомендации
if 'test_recommendations' in locals() and not test_recommendations.empty:
    test_recommendations.to_csv('lab4_results/test_recommendations.csv',
                                encoding='utf-8', index=False)

print("Результаты успешно сохранены в папке 'lab4_results':")
print("  - comparison_results.pkl: результаты сравнения рекомендаций")
print("  - test_recommendations.csv: тестовые рекомендации")

Результаты успешно сохранены в папке 'lab4_results':
  - comparison_results.pkl: результаты сравнения рекомендаций
  - test_recommendations.csv: тестовые рекомендации
